In [1]:
###加载qwen2.5大模型
import os
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_path = "/root/models/Qwen2.5-VL-7B-Instruct/" #"Qwen/Qwen2.5-VL-7B-Instruct"
assert os.path.exists(model_path)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

#模型中每幅图像的视觉标记数量的默认范围为 4-16384。
#您可以根据需要设置 min_pixels 和 max_pixels，例如 256-1280 的 token 范围，以平衡性能和显存占用
processor = AutoProcessor.from_pretrained(
    model_path,
    # min_pixels=256*28*28, max_pixels=1280*28*28,
)

print('\n模型加载完毕！')

ImportError: cannot import name 'Qwen2_5_VLForConditionalGeneration' from 'transformers' (/root/miniconda3/lib/python3.12/site-packages/transformers/__init__.py)

In [ ]:
import json


def read_jsonl_batch(file_path, batch_size):
    """
    按批次读取 JSONL 文件
    :param file_path: JSONL 文件路径
    :param batch_size: 每批数据的大小
    :yield: 每一批数据（列表形式）
    """
    batch = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                parsed_line = json.loads(line.strip())
                batch.append(parsed_line)
                if len(batch) == batch_size:
                    yield batch
                    batch = []
            except json.JSONDecodeError:
                print(f"JSON 解析错误，已跳过该行: {line}")
        if batch:
            yield batch


def identify_attributes(tags, image_path):
    # 这里使用你提供的多模态大模型代码进行属性识别
    # 假设你已经定义了 processor, model, process_vision_info 等
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                },
                {
                    "type": "text",
                    "text": f"""
Objective:
Analyze images and extract visually distinctive attributes for objects belonging to these categories:
{tags}

Attribute Priority (List only what is clearly visible, most distinctive first):

Object type/subcategory (e.g., smartphone, hardcover book)
Color scheme (dominant colors, patterns)
Spatial location (position in the frame)
Appearance (shape, size, notable features)
Material/texture (e.g., glossy, matte, fabric)
Components (attached parts, e.g., strap on handbag)
Interaction Status (how the object is being used, e.g., held, open)
Surface text/graphics (visible logos, labels)
Next to the object (adjacent items)


Rules:
Skip categories not present in the image.
Only list attributes confirmed visually (no guessing).
For multiple instances, describe the most prominent one.
Order attributes by visual distinctiveness (most unique first).

Example Outputs:
[Suitcase]: [hardshell], [navy blue], [on airport cart], [rectangular], [scratches], [wheeled], [stacked with others]
[Laptop]: [gaming laptop], [RGB keyboard], [half-open], [next to mousepad], ["GTX" sticker]
"""
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]


def parse_attributes(attributes_text):
    """
    解析属性文本，将其转换为字典形式
    :param attributes_text: 属性文本
    :return: 包含每个类别属性的字典
    """
    attribute_dict = {}
    lines = attributes_text.strip().split('\n')
    for line in lines:
        if line:
            parts = line.split(':')
            category = parts[0].strip('[]')
            attrs = [attr.strip(' []') for attr in parts[1].split('], [')]
            attribute_dict[category] = attrs
    return attribute_dict


def update_jsonl_batch(data, image_base_path, output_file_path):
    """
    按批次更新 JSONL 文件，将属性添加到 regions 字段
    :param data: 一批数据（列表形式）
    :param image_base_path: 图片文件的基础路径
    :param output_file_path: 输出 JSONL 文件路径
    """
    with open(output_file_path, 'a', encoding='utf-8') as f:
        for item in data:
            filename = item['filename']
            image_path = f"{image_base_path}/{filename}"
            tags = item['tags']
            attributes_text = identify_attributes(tags, image_path)
            attribute_dict = parse_attributes(attributes_text)

            for region in item['grounding']['regions']:
                phrase = region['phrase']
                if phrase in attribute_dict:
                    region['attributes'] = attribute_dict[phrase]

            f.write(json.dumps(item) + '\n')


# 读取 JSONL 文件
file_path = './grounding_data/coco/annotations/instances_train2017_vg_merged6.jsonl'
image_base_path = '/root/assets'
output_file_path = './grounding_data/coco/annotations/instances_train2017_vg_merged6_with_attributes.jsonl'

batch_size = 100  # 每批处理的数据量，可以根据实际情况调整
for batch in read_jsonl_batch(file_path, batch_size):
    update_jsonl_batch(batch, image_base_path, output_file_path)